04_prompt_comparison_gemini.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/github/urcraft/data_analytics_lecture_notebooks/blob/main/04_prompt_comparison_gemini.ipynb

<a target="_blank" href="https://colab.research.google.com/github/urcraft/data_analytics_lecture_notebooks/blob/main/04_prompt_comparison_gemini.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Prompt Comparison: Does How You Ask Change What You Get?

## What you will learn
- Run the **same summarization task** with a naive prompt and a well-crafted prompt.
- Measure the difference with code-based checks.
- Annotate the outputs yourself to judge which prompt produces better summaries.

**NLP Task**: Summarization — condensing longer content into shorter, actionable form.

**Dataset**: `EdinburghNLP/xsum` from HuggingFace — BBC news articles with
single-sentence reference summaries.

**Key question**: Before you reach for a bigger model, have you tried writing a better prompt?

Expected runtime: 10–15 minutes (two prompt runs)
Expected cost: Free-tier Gemini


In [ ]:
%pip install -q google-genai datasets pandas openpyxl


In [ ]:
import os
import json
import time
import pandas as pd

# --- Gemini Setup ---
GEMINI_MODEL = 'gemini-3.1-flash-lite-preview'
print('Using Gemini model:', GEMINI_MODEL)

from google import genai

api_key = os.getenv('GOOGLE_API_KEY')
if not api_key:
    try:
        from google.colab import userdata
        api_key = userdata.get('GOOGLE_API_KEY')
    except Exception:
        api_key = None

if not api_key:
    raise ValueError('Set GOOGLE_API_KEY environment variable (or Colab secret).')

client = genai.Client(api_key=api_key)
print('Gemini client ready.')


## Load Dataset

Same XSum articles used in the summarization notebook.
We use the same seed so results are comparable.


In [ ]:
from datasets import load_dataset

ds = load_dataset('EdinburghNLP/xsum', split='test')

import random
random.seed(42)

candidates = [i for i in range(len(ds)) if len(ds[i]['document']) > 500]
sample_indices = random.sample(candidates, 10)

sample_df = pd.DataFrame([ds[i] for i in sample_indices])
sample_df = sample_df.rename(columns={'document': 'article', 'summary': 'reference_summary'})

print(f'Sample size: {len(sample_df)} articles')
print(f'Average article length: {sample_df["article"].str.len().mean():.0f} characters')


## The Two Prompts

Read both prompts carefully before running them. What differences do you notice?


In [ ]:
# ---- PROMPT A: Naive / Minimal ----
# This is how most people use ChatGPT — short, vague, no constraints.

PROMPT_NAIVE = """Summarize this article:

{article}"""


# ---- PROMPT B: Well-Crafted ----
# This applies several prompt engineering principles:
# - Role assignment ("executive brief writer")
# - Explicit length constraint ("exactly 1-2 sentences")
# - Focus instruction ("most important finding or event")
# - Negative constraint ("do not add information not present")
# - Output format ("Summary:" cue)

PROMPT_CRAFTED = """You are writing an executive brief for a busy decision-maker.
Summarize the following article in exactly 1-2 sentences.
Focus on the single most important finding, event, or outcome.
Do not add any information not present in the article.
Do not include background context — only the key takeaway.

Article:
\"\"\"{article}\"\"\"

Summary:"""


print('Prompt A (Naive):   ', len(PROMPT_NAIVE.split()), 'words (template)')
print('Prompt B (Crafted): ', len(PROMPT_CRAFTED.split()), 'words (template)')


In [ ]:
def summarize(article: str, prompt_template: str) -> dict:
    """Summarize an article using the given prompt template."""
    prompt = prompt_template.format(article=article)
    try:
        resp = client.models.generate_content(model=GEMINI_MODEL, contents=prompt)
        return {'summary': resp.text.strip(), 'error': None}
    except Exception as e:
        return {'summary': None, 'error': str(e)}


---
## Run Prompt A (Naive)


In [ ]:
print(f'Running Prompt A (naive) on {len(sample_df)} articles...\n')

naive_results = []
for _, row in sample_df.iterrows():
    result = summarize(row['article'], PROMPT_NAIVE)
    naive_results.append(result)
    time.sleep(0.5)

sample_df['summary_naive'] = [r['summary'] for r in naive_results]
print('Prompt A complete.')


## Run Prompt B (Well-Crafted)


In [ ]:
print(f'Running Prompt B (well-crafted) on {len(sample_df)} articles...\n')

crafted_results = []
for _, row in sample_df.iterrows():
    result = summarize(row['article'], PROMPT_CRAFTED)
    crafted_results.append(result)
    time.sleep(0.5)

sample_df['summary_crafted'] = [r['summary'] for r in crafted_results]
print('Prompt B complete.')


---
## Code-Based Checks

We measure three things that a good executive summary should have:
- **Conciseness**: short word count (the whole point of summarizing)
- **Brevity**: low sentence count (we asked for 1-2 sentences)
- **Content grounding**: high overlap with the article's vocabulary (suggests it's not hallucinating)


In [ ]:
def word_count(text):
    return len(text.split()) if text else 0

def sentence_count(text):
    if not text:
        return 0
    return len([s for s in text.replace('!', '.').replace('?', '.').split('.') if s.strip()])

def content_overlap(article, summary):
    """Fraction of content words in the summary that also appear in the article."""
    if not summary or not article:
        return 0.0
    stop = {'the','a','an','is','are','was','were','in','on','at','to','for','of',
            'and','or','but','with','that','this','it','has','have','had','be','been'}
    article_words = set(article.lower().split()) - stop
    summary_words = set(summary.lower().split()) - stop
    if not summary_words:
        return 0.0
    return len(summary_words & article_words) / len(summary_words)

# Compute metrics for both prompts
for label, col in [('naive', 'summary_naive'), ('crafted', 'summary_crafted')]:
    sample_df[f'wc_{label}'] = sample_df[col].apply(word_count)
    sample_df[f'sc_{label}'] = sample_df[col].apply(sentence_count)
    sample_df[f'overlap_{label}'] = sample_df.apply(
        lambda r: content_overlap(r['article'], r[col]), axis=1
    )

print('=== Code-Based Comparison ===')
print(f'{"Metric":<30} {"Prompt A (Naive)":>18} {"Prompt B (Crafted)":>18}')
print('-' * 68)
print(f'{"Avg word count":<30} {sample_df["wc_naive"].mean():>15.1f}    {sample_df["wc_crafted"].mean():>15.1f}')
print(f'{"Avg sentence count":<30} {sample_df["sc_naive"].mean():>15.1f}    {sample_df["sc_crafted"].mean():>15.1f}')
print(f'{"Avg content overlap":<30} {sample_df["overlap_naive"].mean():>14.0%}     {sample_df["overlap_crafted"].mean():>14.0%}')
print(f'{"Summaries ≤60 words":<30} {(sample_df["wc_naive"] <= 60).mean():>14.0%}     {(sample_df["wc_crafted"] <= 60).mean():>14.0%}')
print(f'{"Summaries ≤2 sentences":<30} {(sample_df["sc_naive"] <= 2).mean():>14.0%}     {(sample_df["sc_crafted"] <= 2).mean():>14.0%}')


## Side-by-Side Comparison

Read through each triple carefully: the reference summary, the naive output, and the crafted output.


In [ ]:
for idx, row in sample_df.iterrows():
    print(f'=== Article {idx} ===')
    print(f'Reference:  {row["reference_summary"]}')
    print(f'Naive:      {row["summary_naive"]}')
    print(f'Crafted:    {row["summary_crafted"]}')
    print(f'  [Naive: {row["wc_naive"]}w / {row["sc_naive"]}s  |  Crafted: {row["wc_crafted"]}w / {row["sc_crafted"]}s]')
    print()


## Discussion Questions

1. **Did the well-crafted prompt produce more concise summaries?**
   Look at the word counts. The naive prompt often produces paragraph-length responses
   because it has no length constraint.

2. **Did either prompt hallucinate?** Read the summaries and check:
   did the LLM add facts not present in the article? This is easier to catch
   when the summary is short (fewer words = fewer places to hide errors).

3. **Which prompt captures the key point better?**
   Sometimes a longer, rambling summary actually mentions the key point
   buried inside it — but a good executive brief leads with it.

4. **The business implication:**
   Before upgrading to a more expensive model, try improving your prompt first.
   A well-crafted prompt on a cheap model often beats a lazy prompt on an expensive one.

## Export for Your Annotation

Download this Excel file. For each article, annotate both summaries:
- **naive_captures_key_point** (YES/NO)
- **crafted_captures_key_point** (YES/NO)
- **naive_has_hallucination** (YES/NO)
- **crafted_has_hallucination** (YES/NO)
- **which_is_better** (NAIVE / CRAFTED / TIE)
- **student_notes**: your observations


In [ ]:
export_df = sample_df[['article', 'reference_summary', 'summary_naive', 'summary_crafted',
                        'wc_naive', 'wc_crafted', 'overlap_naive', 'overlap_crafted']].copy()

# Truncate article for readability in Excel
export_df['article_excerpt'] = export_df['article'].str[:500] + '...'
export_df = export_df.drop(columns=['article'])

# Annotation columns
export_df['naive_captures_key_point'] = ''
export_df['crafted_captures_key_point'] = ''
export_df['naive_has_hallucination'] = ''
export_df['crafted_has_hallucination'] = ''
export_df['which_is_better'] = ''
export_df['student_notes'] = ''

export_path = 'prompt_comparison_results.xlsx'
export_df.to_excel(export_path, index=False)
print(f'Saved {export_path}')

try:
    from google.colab import files
    files.download(export_path)
except Exception:
    print('Download the file from the notebook working directory.')
